<span style="font-size: 5em">🦜</span>

# __LangGraph Essentials__
## Lab 1: States & Nodes

<img src="../assets/States_Nodes.png" align="left" width="600" style="margin-right:15px;"/>


StateGraph顾名思义，包含状态和图。正如我们刚刚所探讨的，状态就是简单的数据。它被提供给图，由图进行更新，然后再返回给用户。而这些图本身则是无状态的。在定义图时，你首先需要确定图将作用于哪种状态。这种状态会被图中的所有节点共享。
而且，状态通常是一种 Python 数据结构。在我们的示例中，我们将使用一个带有一个字段的Python字典，该字段只是一个字符串列表。因此，当图被调用时，状态会被初始化。在图执行过程中，LangGraph 运行时会选定一个节点来执行，随后它会将当前状态传递给该节点，运行该节点，最后再更新状态。
结合节点的执行结果。节点实际上就是一个函数。你可以看到，它的主要参数是“State”，输出则是对State的更新。在这里，它更新的是我们的字符串列表。状态可以跨时间持久化保存，尤其是在节点发生故障时也能保持不变。因此，举例来说，如果某个节点在执行过程中出现故障，它可以被重新启动，状态得以恢复，然后该函数便可从头开始再次运行。
这需要底层平台提供一些协助来检测故障，但持久化状态是LangGraph的一项关键特性，它能使您的应用更具弹性。

LangGraph 将工作流组织成图，其中节点代表函数，边定义执行流程。所有节点共享一个公共状态，该状态在节点间传递。本示例演示了如何定义状态、创建节点以及将它们连接成一个可执行的图。


In [2]:
from IPython.display import Image, display
import operator
from typing import Annotated, List, Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

In [3]:
class State(TypedDict):
    nlist: List[str]

In [4]:
def node_a(state: State) -> State:
    print(f"node a is receiving {state['nlist']}")
    note = "Hello World from Node a"
    return (State(nlist = [note]))

In [5]:
builder = StateGraph(State)
builder.add_node("a", node_a)
builder.add_edge(START, "a")
builder.add_edge("a", END)
graph = builder.compile()

In [10]:
# https://mermaid.live/
# display(Image(graph.get_graph().draw_mermaid_png()))

# 输出 Mermaid 源码
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	a(a)
	__end__([<p>__end__</p>]):::last
	__start__ --> a;
	a --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
initial_state = State(
    nlist = ["Hello Node a, how are you?"]
)
graph.invoke(initial_state)

node a is receiving ['Hello Node a, how are you?']


{'nlist': ['Hello World from Node a']}

## 要点

Setup:

- State: 所有节点共享相同的状态，该状态可以是 Python TypedDict、dataclass 或 Pydantic BaseModel。
- Nodes: 定义为简单的 Python 函数，接收状态作为输入并返回更新后的状态。

Execution (invoke):

- Runtime: 调用 `invoke` 时，图会根据 `invoke` 语句初始化输入状态，并确定要运行哪些节点。
- State Flow: 每个节点接收当前状态作为输入，执行其逻辑，并返回更新后的状态。
- Graph Return: 所有节点执行完毕后，图返回最终状态值。

Try Next:

- 向图中添加另一个节点并用边连接它。
- 修改节点函数中的打印语句或更改初始状态消息。
- 使用其他字段扩展状态 TypedDict，以便在节点之间传递更多数据。